# Text wrangling and cleaning (pre-processing)



In [4]:
import html
import re
import pandas as pd
import numpy as np

# read a dataset with 100 papers

hcai_paper = pd.read_csv("/Users/jieswang/PycharmProjects/Human_centered_AI_in_papers_and_patents/data/00-open_alex_datasets/hcai_paper_dataset_dropped_duplicates_2026-10-12_15-12-22.csv").sort_values(by='citations', ascending=False).head(100)

In [6]:
hcai_paper.dtypes

id               str
doi              str
title            str
year         float64
citations      int64
authors          str
topic            str
keywords         str
abstract         str
type             str
dtype: object

In [9]:
# inspect abstract
for index, row in hcai_paper.sample(3).iterrows():
    print("\n------ Separator ------")
    print(f"Title: {row['title']}")
    print(f"Author: {row['authors']}")
    print(f"Abstract: {row['abstract']}")


------ Separator ------
Title: Let's Think Together! Assessing Shared Mental Models, Performance, and Trust in Human-Agent Teams
Author: ['Beau G. Schelble', 'Christopher Flathmann', 'Nathan J. McNeese', 'Guo Freeman', 'Rohit Mallick']
Abstract: An emerging research agenda in Computer-Supported Cooperative Work focuses on human-agent teaming and AI agent's roles and effects in modern teamwork. In particular, one understudied key question centers around the construct of team cognition within human-agent teams. This study explores the unique nature of team dynamics in human-agent teams compared to human-human teams and the impact of team composition on perceived team cognition, team performance, and trust. In doing so, a mixed-method approach, including three team composition conditions (all human, human-human-agent, human-agent-agent), completed the team simulation NeoCITIES and completed shared mental model, trust, and perception measures. Results found that human-agent teams are simi

## Cleaning text
Let's clean the title firsts, and they are shorter.

In [12]:

def clean(text: str):
    # convert html escapes like &amp; to characters.
    text = html.unescape(text)
    # tags like <tab>
    text = re.sub(r'<[^<>]*>', ' ', text)
    # markdown URLs like [Some text](https://....)
    text = re.sub(r'\[([^\[\]]*)\]\([^\(\)]*\)', r'\1', text)
    # text or code in brackets like [0]
    text = re.sub(r'\[[^\[\]]*\]', ' ', text)
    # standalone sequences of specials, matches &# but not #cool
    text = re.sub(r'(?:^|\s)[&#<>{}\[\]+|\\:-]{1,}(?:\s|$)', ' ', text)
    # standalone sequences of hyphens like --- or ==
    text = re.sub(r'(?:^|\s)[\-=\+]{2,}(?:\s|$)', ' ', text)
    # sequences of white spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

hcai_paper['title_cleaned'] = hcai_paper['title'].apply(clean).str.lower()

In [13]:
hcai_paper[['title','title_cleaned']]

,title,title_cleaned
7108,Causability and explainability of artificial i...,causability and explainability of artificial i...
7170,Industry 5.0—A Human-Centric Solution,industry 5.0—a human-centric solution
7047,Human-Centered Artificial Intelligence: Reliab...,human-centered artificial intelligence: reliab...
5942,What Is Qualitative Research? An Overview and ...,what is qualitative research? an overview and ...
6509,Practical and ethical challenges of large lang...,practical and ethical challenges of large lang...
...,...,...
6787,Smooth and Resilient Human–Machine Teamwork as...,smooth and resilient human–machine teamwork as...
7072,Geometry Guided Pose-Invariant Facial Expressi...,geometry guided pose-invariant facial expressi...
6753,Human-Centered Artificial Intelligence: The Su...,human-centered artificial intelligence: the su...
6679,Utopia versus dystopia: Professional perspecti...,utopia versus dystopia: professional perspecti...


## Sentence splitter

In [14]:
from nltk.tokenize import sent_tokenize

random_abstract = hcai_paper.sample(1).iloc[0]['abstract']
print(random_abstract)

Well-designed technologies that offer high levels of human control and high levels of computer automation can increase human performance, leading to wider adoption. The Human-Centered Artificial Intelligence (HCAI) framework clarifies how to (1) design for high levels of human control and high levels of computer automation so as to increase human performance, (2) understand the situations in which full human control or full computer control are necessary, and (3) avoid the dangers of excessive human control or excessive computer control. The methods of HCAI are more likely to produce designs that are Reliable, Safe & Trustworthy (RST). Achieving these goals will dramatically increase human performance, while supporting human self-efficacy, mastery, creativity, and responsibility.


In [16]:
all_sent = sent_tokenize(random_abstract)
print(len(all_sent))
print(all_sent)

4
['Well-designed technologies that offer high levels of human control and high levels of computer automation can increase human performance, leading to wider adoption.', 'The Human-Centered Artificial Intelligence (HCAI) framework clarifies how to (1) design for high levels of human control and high levels of computer automation so as to increase human performance, (2) understand the situations in which full human control or full computer control are necessary, and (3) avoid the dangers of excessive human control or excessive computer control.', 'The methods of HCAI are more likely to produce designs that are Reliable, Safe & Trustworthy (RST).', 'Achieving these goals will dramatically increase human performance, while supporting human self-efficacy, mastery, creativity, and responsibility.']


## Normalization
Text normalization is the process of transforming raw text into a single, standardized canonical form so that downstream systems can process it consistently

In [19]:
import textacy.preprocessing as tprep

normalize = tprep.make_pipeline(
    tprep.normalize.hyphenated_words,
    tprep.normalize.quotation_marks,
    tprep.normalize.unicode,
    tprep.remove.accents,
    tprep.remove.punctuation,
    tprep.normalize.whitespace
)

hcai_paper['title_normalized'] = hcai_paper['title_cleaned'].apply(normalize)

In [20]:
display(hcai_paper[['title','title_cleaned','title_normalized']])

,title,title_cleaned,title_normalized
7108,Causability and explainability of artificial i...,causability and explainability of artificial i...,causability and explainability of artificial i...
7170,Industry 5.0—A Human-Centric Solution,industry 5.0—a human-centric solution,industry 5 0 a human centric solution
7047,Human-Centered Artificial Intelligence: Reliab...,human-centered artificial intelligence: reliab...,human centered artificial intelligence reliabl...
5942,What Is Qualitative Research? An Overview and ...,what is qualitative research? an overview and ...,what is qualitative research an overview and g...
6509,Practical and ethical challenges of large lang...,practical and ethical challenges of large lang...,practical and ethical challenges of large lang...
...,...,...,...
6787,Smooth and Resilient Human–Machine Teamwork as...,smooth and resilient human–machine teamwork as...,smooth and resilient human machine teamwork as...
7072,Geometry Guided Pose-Invariant Facial Expressi...,geometry guided pose-invariant facial expressi...,geometry guided pose invariant facial expressi...
6753,Human-Centered Artificial Intelligence: The Su...,human-centered artificial intelligence: the su...,human centered artificial intelligence the sup...
6679,Utopia versus dystopia: Professional perspecti...,utopia versus dystopia: professional perspecti...,utopia versus dystopia professional perspectiv...


## Tokenization

In [39]:
# Using Python string split function
s = ("Hello everyone !\n\nWelcome to EST 610 !")
s.split()

['Hello', 'everyone', '!', 'Welcome', 'to', 'EST', '610', '!']

In [40]:
# Using nltk tokeniers
from nltk import word_tokenize
from nltk.tokenize import regexp_tokenize, wordpunct_tokenize, blankline_tokenize

print("nltk.word_tokenize:", word_tokenize(s))
print("regexp_tokenize (words):", regexp_tokenize(s, pattern=r'\w+')) # \w+ matches sequences of word characters, ignoring spaces and most punctuation.
print("regexp_tokenize (digits):", regexp_tokenize(s, pattern=r'\d+'))
print("wordpunct tokenizer:", wordpunct_tokenize(s))
print("bankline tokenizer", blankline_tokenize(s)) # splits a text into paragraphs based on blank lines (\n\n)

nltk.word_tokenize: ['Hello', 'everyone', '!', 'Welcome', 'to', 'EST', '610', '!']
regexp_tokenize (words): ['Hello', 'everyone', 'Welcome', 'to', 'EST', '610']
regexp_tokenize (digits): ['610']
wordpunct tokenizer: ['Hello', 'everyone', '!', 'Welcome', 'to', 'EST', '610', '!']
bankline tokenizer ['Hello everyone !', 'Welcome to EST 610 !']


In [42]:
# Let's try tokenize our titles in the paper dataset
hcai_paper['title_tokenized'] = hcai_paper['title_normalized'].apply(word_tokenize)

display(hcai_paper[['title','title_tokenized']])

,title,title_tokenized
7108,Causability and explainability of artificial i...,"[causability, and, explainability, of, artific..."
7170,Industry 5.0—A Human-Centric Solution,"[industry, 5, 0, a, human, centric, solution]"
7047,Human-Centered Artificial Intelligence: Reliab...,"[human, centered, artificial, intelligence, re..."
5942,What Is Qualitative Research? An Overview and ...,"[what, is, qualitative, research, an, overview..."
6509,Practical and ethical challenges of large lang...,"[practical, and, ethical, challenges, of, larg..."
...,...,...
6787,Smooth and Resilient Human–Machine Teamwork as...,"[smooth, and, resilient, human, machine, teamw..."
7072,Geometry Guided Pose-Invariant Facial Expressi...,"[geometry, guided, pose, invariant, facial, ex..."
6753,Human-Centered Artificial Intelligence: The Su...,"[human, centered, artificial, intelligence, th..."
6679,Utopia versus dystopia: Professional perspecti...,"[utopia, versus, dystopia, professional, persp..."


## Stemming
(After tokenization)

In [43]:
from nltk.stem import PorterStemmer # import Porter stemmer
from nltk.stem.lancaster import LancasterStemmer
from nltk.stem import SnowballStemmer

pst = PorterStemmer()   # create obj of the PorterStemmer
lst = LancasterStemmer() # create obj of LancasterStemmer
lst.stem("eating")

'eat'

In [44]:

pst.stem("shopping")


'shop'

## Lemmatization

In [46]:
from nltk.stem import WordNetLemmatizer
wlem = WordNetLemmatizer()

# pos='v' to tell NLTK that the word is a verb:
wlem.lemmatize("ate", pos='v')


'eat'

In [49]:
print("lemmatize 'better' as a noun:", wlem.lemmatize("better", pos='n'))
print("lemmatize 'better' as a adjective:", wlem.lemmatize("better", pos='a'))

lemmatize 'better' as a noun: better
lemmatize 'better' as a adjective: good
